# Waveforms, sampling and the DFT

Sampling turns a continuous tone into a list of numbers at a fixed rate. This
notebook builds short sines, square and sawtooth waves, then amplitude and
frequency modulation, and looks at each with a waveform plot, a spectrum and a
player. No prior AudioSpylt tutorial is required.

In [ ]:
from pathlib import Path
import numpy as np
from scipy import signal
import audiospylt as asp

## Sampling a sine

A sine of frequency `f` at sample rate `sr` is `sin(2π·f·t)` with
`t = np.arange(n) / sr`. Here `f = 440 Hz`, `sr = 8000 Hz` (strictly above
`2 × 440 Hz`), duration `0.1 s`, and level −12 dB via `asp.db_to_amplitude`.
`asp.spectrum` is the DFT of the whole signal with a Hann window; its peaks are
the frequencies present. The spectrum should show one peak at 440 Hz; the
waveform and player show the same tone in time.

In [ ]:
sr, f, duration = 8000, 440.0, 0.1
n = int(sr * duration)
t = np.arange(n) / sr
level = asp.db_to_amplitude(-12)
sine = level * np.sin(2 * np.pi * f * t)
print(f"level −12 dB (= {level:.4f}); duration {duration} s, peak |amp| {np.max(np.abs(sine)):.3f}")
print(f"sr={sr} Hz > 2×{f:g} = {2 * f:g} Hz")
asp.nb.show(asp.viz.plot_waveform(sine, sr))
asp.nb.show(asp.viz.plot_spectrum(asp.spectrum(sine, sr)))
asp.nb.play(sine, sr)

## The Nyquist limit

Frequencies above `sr / 2` cannot be represented. A zero-phase sine exactly at
`sr / 2` is `sin(π·n)` at integer sample index `n`, so every sample is 0. The
cell uses `sr = 8000 Hz` and `f = 4000 Hz`.

In [ ]:
sr, f, n = 8000, 4000.0, 64
t = np.arange(n) / sr
nyquist = np.sin(2 * np.pi * f * t)
print(f"max |sample| = {np.max(np.abs(nyquist)):.3e}")
asp.nb.show(asp.viz.plot_waveform(nyquist, sr))
asp.nb.show(asp.viz.plot_spectrum(asp.spectrum(nyquist, sr)))

The printed maximum is `3.920e-14` (rounding error); the tone vanishes.

## Aliasing

A sine above `sr / 2` folds to `|f − k·sr|` for the integer `k` that lands in
`[0, sr / 2]`. Before you run the cell: with `sr = 8000 Hz` and `f = 5000 Hz`,
where does the peak land?

In [ ]:
sr, f, n = 8000, 5000.0, 800
t = np.arange(n) / sr
aliased = np.sin(2 * np.pi * f * t)
spec = asp.spectrum(aliased, sr)
peaks = asp.find_peaks(spec, min_db=-20)
print(f"predicted fold {abs(f - sr):g} Hz; measured {peaks.freq.iloc[0]:.1f} Hz")
asp.nb.show(asp.viz.plot_waveform(aliased, sr))
asp.nb.show(asp.viz.plot_spectrum(spec, peaks))
asp.nb.play(aliased, sr)

The spectrum peak is at 3000 Hz (`|5000 − 8000|`), not at 5000 Hz. The player
sounds a 3000 Hz tone.

## Reading at another rate

Write a 44100 Hz WAV with a 1000 Hz tone and a 5000 Hz tone, then read it back
at 8000 Hz. Before you run the cell: will the 5000 Hz tone fold to 3000 Hz as in
the aliasing example above, or disappear?

In [ ]:
Path("rendered_audio").mkdir(exist_ok=True)
sr_hi = 44100
t_hi = np.arange(int(0.25 * sr_hi)) / sr_hi
tones = np.sin(2 * np.pi * 1000 * t_hi) + 0.5 * np.sin(2 * np.pi * 5000 * t_hi)
tones /= np.max(np.abs(tones))
path = asp.write_audio("rendered_audio/tones_1k_5k.wav", tones, sr_hi)
x8, sr8 = asp.load_audio(path, sr=8000)
spec8 = asp.spectrum(x8, sr8)

def band_db(f, bw=80):
    m = (spec8.freqs >= f - bw) & (spec8.freqs <= f + bw)
    return float(asp.amplitude_to_db(float(spec8.amp[m].max())))

print(f"wrote {path.name} at {sr_hi} Hz; read at {sr8} Hz")
print(f"near 1 kHz: {band_db(1000):.1f} dB; near 3 kHz: {band_db(3000):.1f} dB")
peaks8 = asp.find_peaks(spec8, min_db=-40)
print(peaks8[["freq", "amp"]].round(4).to_string())
asp.nb.show(asp.viz.plot_spectrum(spec8, peaks8))
asp.nb.play(x8, sr8)

The resampler (soxr, high quality) removes the 5000 Hz tone instead of folding
it: near 3000 Hz the reading is −120.0 dB, the lowest value `amplitude_to_db`
returns (its `floor_db`), while 1000 Hz stays at −3.5 dB. That is the opposite
of the aliasing cell above.

## Oscillators

`scipy.signal.square` and `scipy.signal.sawtooth` add harmonics to the same
fundamental. Before you run the cell: which harmonics does each waveform keep
(odd only, or all integers)?

In [ ]:
sr, f, n = 8000, 200.0, 1600
t = np.arange(n) / sr
waves = {"sine": np.sin(2 * np.pi * f * t), "square": signal.square(2 * np.pi * f * t),
         "saw": signal.sawtooth(2 * np.pi * f * t)}
for name, x in waves.items():
    spec = asp.spectrum(x, sr)
    peaks = asp.find_peaks(spec, min_db=-25, min_distance_hz=150, fmax=2000)
    print(name, [int(round(v)) for v in peaks.freq])
    asp.nb.show(asp.viz.plot_waveform(x, sr, title=name))
    asp.nb.show(asp.viz.plot_spectrum(spec, peaks, title=name))
    asp.nb.play(x, sr)

The sine keeps only 200 Hz. The square keeps the odd harmonics (200, 600, 1000,
1400, 1800 Hz). The sawtooth keeps every integer harmonic up to 2000 Hz
(200, 400, …, 2000 Hz).

## Amplitude and frequency modulation

AM is `(1 + m·sin(2π·fm·t))·sin(2π·fc·t)`. FM is
`sin(2π·fc·t + I·sin(2π·fm·t))`. Here `fc = 1000 Hz`, `fm = 200 Hz`, `m = 0.5`,
`I = 2`, `sr = 8000 Hz`, duration 0.5 s. The AM signal is scaled by `1 / (1 + m)`
so its peak stays at full scale. Listen for sidebands around the carrier.

In [ ]:
sr, n, fc, fm, m, I = 8000, 4000, 1000.0, 200.0, 0.5, 2.0
t = np.arange(n) / sr
am = (1 + m * np.sin(2 * np.pi * fm * t)) * np.sin(2 * np.pi * fc * t) / (1 + m)
fm_sig = np.sin(2 * np.pi * fc * t + I * np.sin(2 * np.pi * fm * t))
print(f"m/2 = {m / 2:g} (sideband / carrier amplitude)")
for name, x in [("AM", am), ("FM", fm_sig)]:
    spec = asp.spectrum(x, sr)
    peaks = asp.find_peaks(spec, min_db=-25, min_distance_hz=80, fmin=100, fmax=2500)
    print(name, [int(round(v)) for v in peaks.freq], list(np.round(peaks.amp, 3)))
    asp.nb.show(asp.viz.plot_waveform(x, sr, title=name))
    asp.nb.show(asp.viz.plot_spectrum(spec, peaks, title=name)); asp.nb.play(x, sr)

AM places energy at the carrier and at `fc ± fm` (1000 Hz, plus 800 and
1200 Hz). Each sideband amplitude is `m / 2` of the carrier (here 0.25; printed
amps 0.167 / 0.667 / 0.167). FM sidebands sit at `fc ± k·fm`; with `I = 2` and
the cell's `-25 dB` floor they run from 400 to 1600 Hz (`k` up to 3). Weaker
sidebands continue beyond that floor.

## Mix and write

Sum the AM and FM signals and write a WAV under `rendered_audio/`.

In [ ]:
mix = 0.5 * am + 0.5 * fm_sig
Path("rendered_audio").mkdir(exist_ok=True)
path = asp.write_audio("rendered_audio/am_fm_mix.wav", mix, sr)
print(path, f"{len(mix) / sr:.3f} s, peak |amp|={np.max(np.abs(mix)):.3f}")
asp.nb.show(asp.viz.plot_waveform(mix, sr))
asp.nb.play(mix, sr)

## Window and zero padding

A strong sine between two bins (4.37 cycles in an 800-sample window at 8000 Hz)
plus a weaker neighbour 2.5 bins above it. Compare `window="boxcar"` with the
default Hann, and `zero_pad=1` with the default 4; then once with
`interpolate=False`, which keeps the nearest bin of the zero-padded DFT (here
2.5 Hz apart at zero_pad=4, not the 10 Hz bins of the 800-point transform).
`help(asp.find_peaks)` lists the rest.

In [ ]:
sr, n = 8000, 800
t = np.arange(n) / sr
f1, f2 = 4.37 * sr / n, (4.37 + 2.5) * sr / n
x = np.sin(2 * np.pi * f1 * t) + 0.2 * np.sin(2 * np.pi * f2 * t)
print(f"true strong {f1:.2f} Hz ({4.37} cycles/window), weak {f2:.2f} Hz")
print(f"zero_pad=4 bin spacing {sr / (n * 4):.1f} Hz; zero_pad=1 spacing {sr / n:.1f} Hz")

def err(peaks):
    p = float(peaks.freq.iloc[(peaks.freq - f1).abs().argmin()])
    return p, 1200 * np.log2(p / f1)

for label, peaks in [
    ("boxcar, zero_pad=4", asp.find_peaks(asp.spectrum(x, sr, window="boxcar"), min_db=-40)),
    ("Hann, zero_pad=4", asp.find_peaks(asp.spectrum(x, sr), min_db=-40)),
    ("Hann, zero_pad=1", asp.find_peaks(asp.spectrum(x, sr, zero_pad=1), min_db=-40)),
    ("raw bin", asp.find_peaks(asp.spectrum(x, sr), min_db=-40, interpolate=False)),
]:
    freq, cents = err(peaks)
    print(f"{label}: {freq:.2f} Hz, {cents:+.2f} cents")

Hann at zero_pad=4 is closest (+0.28 cents). With zero_pad=1 the error rises to
+5.66 cents; the raw bin of the 4× zero-padded DFT sits at 42.50 Hz (−48.20
cents), 2.5 Hz from its neighbours. Boxcar under the interpolator is biased
(−7.01 cents here).